# 19 · Apriori — Association Rules in People Analytics

> *"Correlation is what dashboards show you.  
> Association rules show you what conditions tend to occur together —  
> and what outcome tends to follow.  
> That's the difference between a report and a policy."*

---

## 🏢 Business Context

A manufacturing company's HR Analytics team faces a persistent challenge:  
they know that attrition, absenteeism, and low performance are problems.  
They don't know **which employee profiles systematically produce those outcomes**.

Traditional HR analytics answers: *"What percentage of night-shift employees leave?"*  
Association rule mining answers: *"When night shift + no training + temporary contract co-occur,  
how much more likely is attrition than baseline?"*

That's **lift** — the key metric that converts correlations into actionable insight.

This notebook applies the **Apriori algorithm** to **1,500 employee records** across 8 HR dimensions:
department, shift, seniority, contract type, remote work, training status, overtime, and manager rating.  
The outcome variables — attrition, absenteeism, and performance — are included in the transactions,  
allowing the algorithm to discover what antecedent conditions statistically predict each outcome.

The result: a **People Risk Analyzer** that flags any employee profile against the rule library  
and surfaces the most relevant retention and development interventions.

---

**Project:** 19 | **Algorithm:** Apriori / Association Rules | **Domain:** People Analytics / HR  
**Family:** Unsupervised Learning · Frequent Pattern Mining  
**Status:** 📦 Paid Project — [lozanolsa.gumroad.com](https://lozanolsa.gumroad.com)

---

## ⚙️ Section 2 — Setup

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Association Rules ─────────────────────────────────────────────────────────
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# ── Display ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:.3f}".format)

# ── LozanoLsa palette ────────────────────────────────────────────────────────
C_BLUE   = "#4C9BE8"   # primary brand
C_RED    = "#E8574C"   # risk / alert
C_GREEN  = "#3FB950"   # positive / retain
C_AMBER  = "#F0A84D"   # caution / medium risk
C_PURPLE = "#A371F7"   # absenteeism
C_GRAY   = "#8b949e"   # muted

OUTCOME_COLORS = {
    "attrition=Yes":     C_RED,
    "attrition=No":      C_GREEN,
    "performance=High":  C_BLUE,
    "performance=Low":   C_AMBER,
    "absenteeism=High":  C_PURPLE,
    "absenteeism=Low":   C_GREEN,
}

# Aesthetic defaults
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

print("✅ Environment ready — Project 19 · Apriori People Analytics")

## 📂 Section 3 — Load Data

**Dataset:** `people_data_analysis.csv` — 1,500 employee records across 11 HR dimensions.

| Column | Type | Values | Description |
|---|---|---|---|
| `department` | categorical | Operations, Quality, Logistics, Production, Admin | Organizational unit |
| `shift` | categorical | Day, Night | Work schedule |
| `seniority` | categorical | Junior, Mid, Senior | Experience level |
| `contract_type` | categorical | Permanent, Temporary | Employment type |
| `remote_work` | categorical | Yes, No | Remote work access |
| `training_status` | categorical | Completed, Partial, None | Last training cycle |
| `overtime` | categorical | Yes, No | Regular overtime |
| `manager_rating` | categorical | High, Medium, Low | Direct manager assessment |
| `absenteeism` | categorical | Low, Medium, High | **Outcome variable** |
| `performance` | categorical | High, Medium, Low | **Outcome variable** |
| `attrition` | categorical | Yes, No | **Outcome variable** |

> ⚠️ **Unlike supervised ML, all columns — including outcomes — are treated as items.**  
> Apriori discovers which HR profile items co-occur with which outcome items,  
> without specifying a fixed dependent variable.

In [ ]:
# ── Portable loader ──────────────────────────────────────────────────────────
try:
    df = pd.read_csv("people_data_analysis.csv")
except FileNotFoundError:
    df = pd.read_csv(
        "https://raw.githubusercontent.com/LozanoLsa/"
        "Apriori_HR_People/main/people_data_analysis.csv"
    )

HR_FEATURES = [
    "department", "shift", "seniority", "contract_type",
    "remote_work", "training_status", "overtime", "manager_rating"
]
OUTCOMES = ["absenteeism", "performance", "attrition"]

print(f"Dataset shape     : {df.shape}")
print(f"Employee records  : {len(df):,}")
print(f"HR dimensions     : {len(HR_FEATURES)}")
print(f"Outcome variables : {len(OUTCOMES)}")
print(f"Missing values    : {df.isnull().sum().sum()}")
print()
print("Outcome distributions:")
for col in OUTCOMES:
    vc = df[col].value_counts()
    print(f"  {col}: {dict(vc)}")
df.head()

## 🔍 Section 4 — Sanity Checks

For association rule mining, the key pre-flight checks differ from clustering:  
1. **No missing values** — NaN creates phantom items in the transaction matrix
2. **No degenerate categories** — a column with 99% of one value won't generate interesting rules
3. **Baseline outcome rates** — we need to know the base rates to interpret lift correctly
4. **Category cardinality** — high-cardinality columns fragment transactions (fewer co-occurrences)

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
print("── Missing values ───────────────────────────────────────")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("  ✅ No missing values — clean transaction matrix.")
else:
    print(missing[missing > 0])

# ── Category distributions ────────────────────────────────────────────────────
print("\n── Category value counts ────────────────────────────────")
for col in df.columns:
    vc = df[col].value_counts()
    majority_pct = vc.iloc[0] / len(df) * 100
    flag = "⚠️ " if majority_pct > 85 else "  "
    print(f"  {flag}{col:<28}: {dict(vc)}")

# ── Baseline outcome rates (for lift interpretation) ─────────────────────────
print("\n── Baseline outcome rates (denominator for lift) ────────")
baselines = {}
for col in OUTCOMES:
    for val in df[col].unique():
        rate = (df[col] == val).mean()
        key  = f"{col}={val}"
        baselines[key] = rate
        print(f"  {key:<28}: {rate:.4f} ({rate*100:.1f}%)")

print()
print("  → Lift > 1.0 means the rule predicts the outcome more often than chance.")
print("    A rule with lift=1.5 on attrition=Yes means 50% more likely than the 14.9% base rate.")

## 📊 Section 5 — Exploratory Data Analysis

Three visualizations orient us before mining:

1. **Outcome rates by key HR dimension** — which variables show the strongest marginal association?  
   These will generate the most powerful rules.
2. **Cross-tabulation heatmap** — which pairs of HR dimensions show elevated outcome rates?
3. **Transaction item frequency** — which items are frequent enough to survive the support threshold?

In [ ]:
# ── EDA 1 · Attrition rate by HR dimension ────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for ax, col in zip(axes, HR_FEATURES):
    rates = df.groupby(col)["attrition"].apply(lambda x: (x=="Yes").mean()).sort_values(ascending=True)
    baseline = (df["attrition"]=="Yes").mean()
    colors = [C_RED if r > baseline * 1.2 else C_GREEN if r < baseline * 0.8 else C_AMBER
              for r in rates.values]
    ax.barh(rates.index, rates.values * 100, color=colors, alpha=0.85, edgecolor="white")
    ax.axvline(baseline * 100, color="black", lw=1.5, ls="--", alpha=0.7, label=f"Baseline {baseline*100:.1f}%")
    ax.set_xlabel("Attrition Rate (%)", fontsize=8)
    ax.set_title(col.replace("_", " ").title(), fontsize=10, fontweight="bold")
    for i, (cat, rate) in enumerate(rates.items()):
        ax.text(rate * 100 + 0.3, i, f"{rate*100:.1f}%", va="center", fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(axis="x", alpha=0.3)

plt.suptitle("Attrition Rate by HR Dimension — Which Variables Drive Risk?",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("  → Red bars: above-baseline attrition risk")
print("  → Green bars: below-baseline (protective factors)")
print("  → Apriori will find which COMBINATIONS amplify these individual effects.")

In [ ]:
# ── EDA 2 · Bivariate heatmap — training status × manager rating ──────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

pairs = [
    ("training_status", "attrition",    "Yes", "Attrition Rate"),
    ("manager_rating",  "absenteeism",  "High","High Absenteeism Rate"),
    ("remote_work",     "performance",  "High","High Performance Rate"),
]

for ax, (xvar, yvar, target, title) in zip(axes, pairs):
    ct = df.groupby([xvar])[yvar].apply(lambda x: (x==target).mean()).reset_index()
    ct.columns = [xvar, "rate"]
    ct_sorted = ct.sort_values("rate", ascending=False)
    baseline = (df[yvar]==target).mean()
    colors = [C_RED if r > baseline else C_GREEN for r in ct_sorted["rate"]]
    bars = ax.bar(ct_sorted[xvar], ct_sorted["rate"]*100, color=colors, alpha=0.85, edgecolor="white")
    ax.axhline(baseline*100, color="black", lw=1.5, ls="--", alpha=0.7, label=f"Baseline {baseline*100:.1f}%")
    for bar, rate in zip(bars, ct_sorted["rate"]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f"{rate*100:.1f}%", ha="center", fontsize=9)
    ax.set_ylabel(f"{title} (%)", fontsize=10)
    ax.set_title(f"{title} by {xvar.replace('_',' ').title()}",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Marginal Outcome Rates by Key HR Drivers",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("  → Training=None employees have highest attrition.")
print("  → Low manager rating drives highest absenteeism.")
print("  → Remote work access correlates with higher performance.")

In [ ]:
# ── EDA 3 · Transaction item frequency bar chart ──────────────────────────────
# This shows which items are most frequent — items below min_support will be pruned
all_items = []
for _, row in df.iterrows():
    for col, val in zip(df.columns, row):
        all_items.append(f"{col}={val}")

item_freq = pd.Series(all_items).value_counts() / len(df)
top_items = item_freq.head(20).sort_values()

colors_item = []
for item in top_items.index:
    if any(o in item for o in ["attrition","performance","absenteeism"]):
        colors_item.append(C_RED if "Yes" in item or "Low" in item else C_GREEN)
    else:
        colors_item.append(C_BLUE)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_items.index, top_items.values, color=colors_item, alpha=0.85, edgecolor="white")
ax.axvline(0.05, color=C_RED, ls="--", lw=2, label="min_support = 0.04")
for i, (item, freq) in enumerate(top_items.items()):
    ax.text(freq + 0.005, i, f"{freq:.3f}", va="center", fontsize=9)
ax.set_xlabel("Item Support (frequency in dataset)", fontsize=11)
ax.set_title("Top 20 Most Frequent Items — Support Distribution",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print("  → Blue bars: HR profile items (predictors)")
print("  → Green/Red bars: outcome items (consequents)")
print("  → Items to the left of the red line will be pruned by min_support.")

## 🔧 Section 6 — Preprocessing: Transaction Encoding

Apriori requires data in **binary transaction format**:  
each row is a basket (employee record) and each column is a binary item (`col=val`).

**No StandardScaler needed** — this is categorical data.  
The `TransactionEncoder` from `mlxtend` converts the DataFrame into the required binary matrix.

> Unlike clustering (where we exclude target variables), in association rule mining  
> **all columns are items** — including outcomes. This allows the algorithm to discover  
> rules in *both directions*: profile → outcome and outcome → profile.

In [ ]:
# ── Build transaction list ────────────────────────────────────────────────────
def build_transactions(df):
    """Convert DataFrame rows to itemized transaction lists."""
    transactions = []
    for _, row in df.iterrows():
        items = [f"{col}={val}" for col, val in zip(df.columns, row)]
        transactions.append(items)
    return transactions

transactions = build_transactions(df)

# ── TransactionEncoder ────────────────────────────────────────────────────────
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

print(f"Transaction matrix: {df_encoded.shape}")
print(f"  Rows (employees): {df_encoded.shape[0]}")
print(f"  Columns (items)  : {df_encoded.shape[1]}")
print(f"  Sparsity         : {(~df_encoded).mean().mean():.1%} zeros")
print()
print("Sample items (first 8 columns):")
print("  ", df_encoded.columns[:8].tolist())
print()
print("Example row (employee 0):")
active = df_encoded.columns[df_encoded.iloc[0]].tolist()
print(f"  Active items: {active}")

## 🔢 Section 7 — Threshold Selection: Support × Confidence

Apriori has two key thresholds:

| Parameter | Role | Trade-off |
|---|---|---|
| **min_support** | Minimum co-occurrence frequency | Low = more rules, many trivial; High = fewer, more common |
| **min_confidence** | Minimum P(consequent \| antecedent) | Low = more rules; High = stricter predictive accuracy |

We sweep support from 0.04 to 0.20 to find the sweet spot:  
enough rules to be useful, few enough to be interpretable.

In [ ]:
# ── Support sweep ─────────────────────────────────────────────────────────────
support_values = [0.04, 0.05, 0.06, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20]
sweep_results = []

for minsup in support_values:
    fi_tmp = apriori(df_encoded, min_support=minsup, use_colnames=True, max_len=4)
    rules_tmp = association_rules(fi_tmp, metric="lift", min_threshold=1.0)
    rules_tmp = rules_tmp[rules_tmp["consequents"].apply(lambda x: len(x) == 1)]
    sweep_results.append({
        "min_support": minsup,
        "n_itemsets": len(fi_tmp),
        "n_rules": len(rules_tmp),
        "max_lift": rules_tmp["lift"].max() if len(rules_tmp) else 0,
        "mean_confidence": rules_tmp["confidence"].mean() if len(rules_tmp) else 0,
    })

sweep_df = pd.DataFrame(sweep_results)
print("── Support sweep results ─────────────────────────────────")
display(sweep_df.set_index("min_support").round(3))

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(sweep_df["min_support"], sweep_df["n_rules"], "o-",
         color=C_BLUE, lw=2.5, ms=9)
ax1.axvline(0.04, color=C_RED, ls="--", lw=1.8, label="Chosen: 0.04")
ax1.set_xlabel("min_support", fontsize=11)
ax1.set_ylabel("Number of HR Rules", fontsize=11)
ax1.set_title("Rule Count vs. min_support", fontsize=12, fontweight="bold")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.plot(sweep_df["min_support"], sweep_df["max_lift"], "s-",
         color=C_AMBER, lw=2.5, ms=9)
ax2.axvline(0.04, color=C_RED, ls="--", lw=1.8, label="Chosen: 0.04")
ax2.set_xlabel("min_support", fontsize=11)
ax2.set_ylabel("Max Lift in Ruleset", fontsize=11)
ax2.set_title("Max Lift vs. min_support", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle(
    "Apriori Threshold Selection — min_support = 0.04, min_confidence = 0.60",
    fontsize=12, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

print("\n── Selection rationale ──────────────────────────────────")
print("  min_support = 0.04 → rules cover ≥60 of 1,500 employees")
print("  Preserves rare-but-impactful patterns (e.g., high-risk combinations)")
print("  Paired with min_lift > 1.0 to filter trivial/spurious associations")

## 🤖 Section 8 — Model Training: Frequent Itemsets + Association Rules

The Apriori pipeline has two steps:

**Step 1 — Frequent Itemsets:**  
`apriori()` finds all item combinations that appear in ≥ `min_support` of records.  
Prunes the search space using the Apriori property:  
*if an itemset is infrequent, all its supersets are also infrequent.*

**Step 2 — Association Rules:**  
`association_rules()` derives directional rules from frequent itemsets:  
**{antecedents} → {consequent}** with support, confidence, and lift.

**Key metrics:**
- **Support** = P(A ∪ B) — how common is the full rule in the data
- **Confidence** = P(B | A) — how often does B occur given A
- **Lift** = P(B | A) / P(B) — how much more likely is B given A vs. by chance  
  → lift > 1: A is associated with B · lift < 1: A is protective against B

In [ ]:
# ── Step 1: Frequent itemsets ────────────────────────────────────────────────
MIN_SUPPORT    = 0.04
MIN_CONFIDENCE = 0.60
MAX_LEN        = 4

frequent_itemsets = apriori(
    df_encoded,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_LEN
)

print(f"── Frequent itemsets ─────────────────────────────────────")
print(f"  Total itemsets: {len(frequent_itemsets)}")
size_dist = frequent_itemsets["itemsets"].apply(len).value_counts().sort_index()
for sz, cnt in size_dist.items():
    print(f"  Size {sz}: {cnt:>6} itemsets")

# ── Step 2: Association rules ────────────────────────────────────────────────
all_rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

# Filter to single-consequent rules and add conviction metric
rules = all_rules[all_rules["consequents"].apply(lambda x: len(x) == 1)].copy()

# Add conviction
rules["conviction"] = (1 - rules["consequents"].map(
    lambda x: df_encoded[list(x)[0]].mean()
)) / (1 - rules["confidence"] + 1e-10)

rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)

print(f"\n── Association rules ────────────────────────────────────")
print(f"  Total rules (single-consequent): {len(rules)}")
print(f"  Lift range    : {rules['lift'].min():.3f} – {rules['lift'].max():.3f}")
print(f"  Confidence range: {rules['confidence'].min():.3f} – {rules['confidence'].max():.3f}")
print(f"  Support range : {rules['support'].min():.3f} – {rules['support'].max():.3f}")

# HR-outcome rules
CRITICAL_OUTCOMES = [
    "attrition=Yes", "attrition=No",
    "performance=High", "performance=Low",
    "absenteeism=High", "absenteeism=Low",
]
hr_rules = rules[rules["consequents"].apply(
    lambda x: any(c in CRITICAL_OUTCOMES for c in list(x))
)].copy()
print(f"  HR-critical rules (outcome consequents): {len(hr_rules)}")

In [ ]:
# ── Lift-Support scatter — rule quality map ────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))

# Background rules (non-outcome)
other_rules = rules[~rules["consequents"].apply(
    lambda x: any(c in CRITICAL_OUTCOMES for c in list(x))
)]
ax.scatter(other_rules["support"], other_rules["lift"],
           c=C_GRAY, alpha=0.18, s=12, edgecolors="none", label="Other rules")

# Outcome rules colored by type
for outcome, color in OUTCOME_COLORS.items():
    sub = hr_rules[hr_rules["consequents"].apply(lambda x: outcome in x)]
    if len(sub):
        ax.scatter(sub["support"], sub["lift"],
                   c=color, alpha=0.8, s=40, edgecolors="white", lw=0.5,
                   label=outcome)

ax.axhline(1.0, color="gray", lw=1, ls="--", alpha=0.7)
ax.set_xlabel("Support (frequency in data)", fontsize=11)
ax.set_ylabel("Lift (vs. baseline)", fontsize=11)
ax.set_title(
    "Lift vs. Support Map — HR Outcome Rules\n"
    "Upper-right = high-frequency, strong-effect rules",
    fontsize=12, fontweight="bold"
)
ax.legend(loc="upper right", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🧠 Section 9 — Rule Profiling & Business Interpretation

We organize the extracted rules into six HR outcome categories and rank by lift.  
Lift is the primary sorting criterion — it measures how much the antecedent conditions  
change the probability of the outcome relative to its unconditional base rate.

In [ ]:
# ── Top rules by outcome category ─────────────────────────────────────────────
categories = {
    "🔴 Attrition Risk":   "attrition=Yes",
    "🟢 Retention Factors": "attrition=No",
    "⭐ High Performance": "performance=High",
    "⚠️  Low Performance":  "performance=Low",
    "📉 High Absenteeism": "absenteeism=High",
    "✅ Low Absenteeism":  "absenteeism=Low",
}

TOP_N = 5

def fmt_items(itemset):
    return " + ".join(sorted([i.replace("_", " ") for i in itemset]))

for label, outcome in categories.items():
    sub = hr_rules[hr_rules["consequents"].apply(lambda x: outcome in x)]
    top = sub.nlargest(TOP_N, "lift")
    print(f"\n{label} — Top {TOP_N} rules (n={len(sub)} total):")
    print(f"  {'Antecedents':<55} {'Supp':>6} {'Conf':>6} {'Lift':>6}")
    print("  " + "─" * 75)
    for _, r in top.iterrows():
        ant = fmt_items(r["antecedents"])
        print(f"  {ant:<55} {r['support']:>6.3f} {r['confidence']:>6.3f} {r['lift']:>6.3f}")

In [ ]:
# ── Lift comparison chart — top rules per outcome ─────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, (label, outcome) in zip(axes, categories.items()):
    color = OUTCOME_COLORS[outcome]
    sub = hr_rules[hr_rules["consequents"].apply(lambda x: outcome in x)]
    top = sub.nlargest(5, "lift")

    if len(top) == 0:
        ax.text(0.5, 0.5, "No rules found", ha="center", va="center", transform=ax.transAxes)
        continue

    ant_labels = ["\n+ ".join([i.split("=")[1] for i in sorted(list(r["antecedents"]))[:3]])
                  for _, r in top.iterrows()]
    bars = ax.barh(range(len(top)), top["lift"].values,
                   color=color, alpha=0.85, edgecolor="white")
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(ant_labels, fontsize=7.5)
    ax.axvline(1.0, color="gray", lw=1, ls="--", alpha=0.7)
    ax.set_xlabel("Lift", fontsize=9)
    ax.set_title(label, fontsize=10, fontweight="bold", color=color)
    for bar, lift in zip(bars, top["lift"]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f"{lift:.3f}", va="center", fontsize=8)
    ax.grid(axis="x", alpha=0.3)

plt.suptitle("Top 5 Rules by Lift — Each Outcome Category",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Key rule summary table ─────────────────────────────────────────────────────
key_rules_data = []
for label, outcome in categories.items():
    sub = hr_rules[hr_rules["consequents"].apply(lambda x: outcome in x)]
    if len(sub) == 0:
        continue
    best = sub.nlargest(1, "lift").iloc[0]
    baseline = baselines[outcome]
    implied_rate = best["confidence"]
    key_rules_data.append({
        "Outcome":    outcome,
        "Category":   label,
        "Antecedents": fmt_items(best["antecedents"]),
        "Support":    round(best["support"], 3),
        "Confidence": round(best["confidence"], 3),
        "Lift":       round(best["lift"], 3),
        "Base Rate":  round(baseline, 3),
        "Implied Rate": round(implied_rate, 3),
    })

summary_df = pd.DataFrame(key_rules_data)
print("── Best rule per outcome category ───────────────────────")
display(summary_df[["Category","Antecedents","Support","Confidence","Lift","Base Rate"]].set_index("Category"))

## ✅ Section 10 — Rule Validation & Quality Assessment

Apriori validation in an HR context uses four layers:

| Validation | Method | What it tells us |
|---|---|---|
| **Rule strength** | Lift distribution by outcome | Are rules above trivial? |
| **Coverage** | Support × N employees | How many employees does each rule affect? |
| **Conviction** | 1 − P(B) / (1 − conf) | How deterministic is the rule? |
| **Business alignment** | Base rate comparison | Does the rule explain meaningful uplift? |

In [ ]:
# ── Rule quality metrics ──────────────────────────────────────────────────────
print("── Rule quality by outcome category ─────────────────────")
print(f"{'Category':<30} {'Count':>6} {'Lift avg':>9} {'Lift max':>9} {'Conf avg':>9} {'Coverage':>9}")
print("─" * 75)
for label, outcome in categories.items():
    sub = hr_rules[hr_rules["consequents"].apply(lambda x: outcome in x)]
    if len(sub) == 0:
        continue
    coverage = (sub["support"] * len(df)).max()
    print(f"  {label:<28} {len(sub):>6} {sub['lift'].mean():>9.3f} {sub['lift'].max():>9.3f} "
          f"{sub['confidence'].mean():>9.3f} {coverage:>8.0f} emp")

# ── Conviction distribution ────────────────────────────────────────────────────
print("\n── Top 10 rules by conviction (determinism) ─────────────")
print(f"  {'Antecedents':<50} {'→ Consequent':<25} {'Conviction':>10} {'Lift':>6}")
print("  " + "─" * 93)
top_conv = hr_rules.nlargest(10, "conviction")
for _, r in top_conv.iterrows():
    ant = fmt_items(r["antecedents"])
    cons = list(r["consequents"])[0]
    conv = min(r["conviction"], 99.99)
    print(f"  {ant:<50} → {cons:<23} {conv:>10.2f} {r['lift']:>6.3f}")

# ── Business uplift check ──────────────────────────────────────────────────────
print("\n── Business uplift — best rule per risk category ─────────")
for outcome in ["attrition=Yes", "absenteeism=High", "performance=Low"]:
    sub = hr_rules[hr_rules["consequents"].apply(lambda x: outcome in x)]
    if len(sub) == 0:
        continue
    best = sub.nlargest(1, "lift").iloc[0]
    base = baselines[outcome]
    implied = best["confidence"]
    uplift  = implied / base
    n_emp   = int(best["support"] * len(df))
    print(f"  {outcome}:")
    print(f"    Base rate: {base:.1%} → Rule confidence: {implied:.1%} (×{uplift:.2f} uplift)")
    print(f"    Covers: {n_emp} employees, Lift: {best['lift']:.3f}")
    print(f"    Rule: {fmt_items(best['antecedents'])}")
    print()

## 🧪 Section 11 — People Risk Analyzer & HR Playbook

Given any employee profile, the rule library can:
1. Identify all matching antecedent rules
2. Rank by lift to surface the most relevant risk/opportunity signals
3. Return a prioritized HR intervention recommendation

**Three representative profiles:**

| Profile | Characteristics | Expected signals |
|---|---|---|
| **A — High risk** | Night + Junior + Temporary + No training + No remote | Attrition + absenteeism risk |
| **B — Medium risk** | Day + Mid + Permanent + Partial training + Overtime | Absenteeism risk |
| **C — Low risk** | Day + Senior + Permanent + Completed + Remote + High mgr | High performance + low attrition |

In [ ]:
# ── People Risk Analyzer ──────────────────────────────────────────────────────
HR_INTERVENTIONS = {
    "attrition=Yes": [
        "Conduct a stay interview within 30 days.",
        "Review compensation and career path alignment.",
        "Evaluate night-shift rotation or hybrid schedule options.",
        "Assign a senior mentor for the first 90 days.",
    ],
    "absenteeism=High": [
        "Implement a flexible schedule or partial remote option.",
        "Review manager feedback scores — low manager rating is a primary driver.",
        "Refer to Employee Assistance Program (EAP) if needed.",
        "Check overtime hours — burnout threshold exceeded?",
    ],
    "performance=Low": [
        "Enroll in next available training cycle immediately.",
        "Assign weekly 1:1 coaching with manager.",
        "Review workload — understaffing elevates junior error rates.",
        "Consider temporary assignment to a senior buddy system.",
    ],
    "performance=High": [
        "Flag as high-potential — ensure succession plan visibility.",
        "Recognize and reward — delay is a retention risk.",
        "Offer stretch assignment or cross-department project.",
    ],
    "absenteeism=Low": [
        "Profile is a benchmark — identify and share best practices.",
        "Consider as internal trainer or culture champion.",
    ],
    "attrition=No": [
        "Profile is a retention anchor — maintain current conditions.",
        "Ensure remote + training access is not threatened by budget cuts.",
    ],
}

def analyze_employee(profile: dict, top_n: int = 5, verbose: bool = True) -> pd.DataFrame:
    """
    Match an employee profile against the HR rule library.
    Returns top matching rules ranked by lift.
    """
    items = frozenset(f"{k}={v}" for k, v in profile.items())
    matched = []
    for _, r in hr_rules.iterrows():
        if r["antecedents"].issubset(items):
            matched.append(r)

    if not matched:
        if verbose:
            print("  No rules matched this profile.")
        return pd.DataFrame()

    matched_df = pd.DataFrame(matched).nlargest(top_n, "lift")

    if verbose:
        print(f"{'─' * 60}")
        print(f"  Matching rules: {len(matched)} | Showing top {top_n} by lift")
        print(f"{'─' * 60}")
        for _, r in matched_df.iterrows():
            cons   = list(r["consequents"])[0]
            ant    = fmt_items(r["antecedents"])
            icon   = {"attrition=Yes": "🔴", "attrition=No": "🟢",
                      "performance=High": "⭐", "performance=Low": "⚠️ ",
                      "absenteeism=High": "📉", "absenteeism=Low": "✅"}.get(cons, "")
            print(f"  {icon} {cons}")
            print(f"     Antecedents : {ant}")
            print(f"     Lift={r['lift']:.3f} | Conf={r['confidence']:.3f} | Supp={r['support']:.3f}")

        # HR intervention
        risk_cons = matched_df["consequents"].apply(lambda x: list(x)[0]).tolist()
        priority_risks = [c for c in risk_cons if c in ["attrition=Yes", "absenteeism=High", "performance=Low"]]
        if priority_risks:
            top_risk = priority_risks[0]
            print(f"\n  TOP RISK: {top_risk}")
            print("  Recommended actions:")
            for i, act in enumerate(HR_INTERVENTIONS.get(top_risk, []), 1):
                print(f"    {i}. {act}")
        print()

    return matched_df

In [ ]:
# ── Profile A: High risk ──────────────────────────────────────────────────────
print("═" * 60)
print("  PROFILE A — HIGH RISK EMPLOYEE")
print("  Night shift · Junior · Temporary · No training")
print("  No remote work · Overtime · Low manager rating")
print("═" * 60)
analyze_employee({
    "shift": "Night", "seniority": "Junior", "contract_type": "Temporary",
    "training_status": "None", "remote_work": "No", "overtime": "Yes",
    "manager_rating": "Low", "department": "Operations"
})

In [ ]:
# ── Profile B: Medium risk ────────────────────────────────────────────────────
print("═" * 60)
print("  PROFILE B — MEDIUM RISK EMPLOYEE")
print("  Day shift · Mid seniority · Permanent · Partial training")
print("  No remote · Overtime · Medium manager rating")
print("═" * 60)
analyze_employee({
    "shift": "Day", "seniority": "Mid", "contract_type": "Permanent",
    "training_status": "Partial", "remote_work": "No", "overtime": "Yes",
    "manager_rating": "Medium", "department": "Production"
})

In [ ]:
# ── Profile C: Low risk / High retention ──────────────────────────────────────
print("═" * 60)
print("  PROFILE C — LOW RISK / HIGH RETENTION PROFILE")
print("  Day shift · Senior · Permanent · Completed training")
print("  Remote work · No overtime · High manager rating")
print("═" * 60)
analyze_employee({
    "shift": "Day", "seniority": "Senior", "contract_type": "Permanent",
    "training_status": "Completed", "remote_work": "Yes", "overtime": "No",
    "manager_rating": "High", "department": "Quality"
})

---

## 💡 Final Reflection

Association rule mining in HR is fundamentally different from regression or classification:  
it doesn't build a model — it builds a **pattern library** that explains relationships
without a fixed dependent variable, from any direction in the data.

Key takeaways from Project 19:

1. **Lift is the HR metric that matters, not correlation.**  
   A rule with lift=1.7 on high absenteeism means that employees with that profile  
   are 70% more likely to be absent than the workforce average.  
   A regression coefficient doesn't tell you that cleanly.

2. **The combination is what creates the risk — not the factor alone.**  
   Night shift alone doesn't predict attrition. Night + Junior + No training does.  
   HR policies that target single variables miss the interaction effect.

3. **Rules work in both directions.**  
   Remote + Completed training → Low absenteeism (lift=1.63) is as useful as  
   No training + No remote → High absenteeism.  
   The protective rules define the retention conditions worth preserving.

4. **Support defines policy scale.**  
   A rule with support=0.04 affects 60 employees.  
   A rule with support=0.15 affects 225 employees.  
   High-lift, high-support rules are the ones worth scaling into policy.

5. **The rule library is not a prediction — it's a diagnostic.**  
   Use it to flag profiles in onboarding, mid-year reviews, and promotion cycles.  
   The intervening actions are what change the outcome.

---

**What to try next:**
- Combine Apriori with **K-Means** (Project 15) — cluster employees first, then mine rules per cluster
- Apply to **time-series HR data** — do the same rules hold across fiscal quarters?
- Deploy as a **real-time onboarding risk scorer** via a simple API

---

*LozanoLsa | Operational Excellence · MBB · Machine Learning | GitHub: LozanoLsa*